# A1.2 — Huấn luyện 8 thí nghiệm trên Google Colab

Notebook này chạy đủ 8 cấu hình (ResNet-18 / DeiT-Tiny × scratch / head / partial / full) của bài A1 trên GPU Colab.

**Cách dùng**
1. Menu **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**.
2. Log và checkpoint được lưu vào Google Drive (`MyDrive/CO5085_A1/runs`). Nếu Colab bị ngắt, chỉ cần mở lại notebook và **Run all**: các thí nghiệm đã xong sẽ được bỏ qua, thí nghiệm đang dở sẽ chạy tiếp từ epoch cuối cùng.
3. Khi cả 8 thí nghiệm xong, chạy ô cuối để tải file `A1_runs.zip` về máy và gửi lại để phân tích.

Thứ tự chạy: 6 cấu hình pretrained (10 epoch) trước, 2 cấu hình from-scratch (30 epoch) sau. Ước tính tổng cộng khoảng 4–6 giờ GPU T4, nên cần 2–3 phiên Colab.

In [ ]:
# Check that a GPU is attached (should show a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Mount Google Drive: logs and checkpoints are written there so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/CO5085_A1/runs"
DATA_DIR = "/content/drive/MyDrive/CO5085_A1/data"
!mkdir -p "$RUN_DIR" "$DATA_DIR"

In [ ]:
# Get the latest code from GitHub
REPO = "/content/Deep_Learning_Computer_Vision_CO5085"
!if [ -d "$REPO" ]; then git -C "$REPO" pull -q; else git clone -q https://github.com/dqpinfo/Deep_Learning_Computer_Vision_CO5085.git "$REPO"; fi
%cd {REPO}/Assignment_1
!git log --oneline -1

# Keep the CIFAR-100 cache on Drive so it is downloaded only once
!rm -rf data && ln -s "$DATA_DIR" data

In [ ]:
# Colab already has torch, torchvision, scikit-learn and pyarrow; only timm is missing
!pip install -q timm

In [ ]:
# Quick pipeline check on 512 images (about 1 minute); not used for reported results
!python train.py --exp resnet18_head --subset 512 --epochs 1 --workers 2

In [ ]:
# Run all 8 experiments. Rerun this cell after a disconnect: it skips finished runs and resumes the interrupted one.
!python run_all.py --run-dir "$RUN_DIR" --workers 2

In [ ]:
# Summary of the finished runs
import json, glob, os
rows = []
for path in sorted(glob.glob(f"{RUN_DIR}/*.json")):
    if path.endswith("_test_preds.json"):
        continue
    log = json.load(open(path))
    rows.append((log["experiment"]["name"], log["test_acc"], log["test_macro_f1"], log["params_trainable"],
                 log["train_time_sec"] / 60, log.get("peak_gpu_mem_mb", 0)))
print(f"{'experiment':20s} {'test acc':>9s} {'macro-F1':>9s} {'trainable':>11s} {'train min':>10s} {'GPU MB':>8s}")
for r in rows:
    print(f"{r[0]:20s} {r[1]*100:9.2f} {r[2]*100:9.2f} {r[3]:11,d} {r[4]:10.1f} {r[5]:8.0f}")
print(f"\n{len(rows)}/8 experiments finished")

In [ ]:
# Download the logs and test predictions (without the large .pth weights)
!cd "$RUN_DIR" && zip -q -j /content/A1_runs.zip *.json
from google.colab import files
files.download("/content/A1_runs.zip")